# Interactive plots: Acceleration RMS vs TIMESTAMP

This notebook scans `data/*.csv` and plots only `Acceleration RMS` against `TIMESTAMP` for each CSV.

Zoom and pan are enabled via `%matplotlib notebook`.

In [ ]:
%matplotlib notebook

from pathlib import Path
import os
import matplotlib.pyplot as plt
import pandas as pd

print('Jupyter cwd:', os.getcwd(), flush=True)

DATA_DIR = Path('data')
print('DATA_DIR exists:', DATA_DIR.exists(), flush=True)

csv_files = sorted(DATA_DIR.glob('*.csv'))
print(f'Found {len(csv_files)} CSV file(s) in {DATA_DIR.resolve()}', flush=True)

for f in csv_files:
    print('-', f.name, flush=True)


In [ ]:
if not csv_files:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR.resolve()}')

required_cols = {'TIMESTAMP', 'Acceleration RMS'}

for csv_path in csv_files:
    print(f'Plotting: {csv_path.name}', flush=True)
    df = pd.read_csv(csv_path)

    missing = required_cols - set(df.columns)
    if missing:
        print(f'  Skipping {csv_path.name}: missing columns {sorted(missing)}', flush=True)
        continue

    ts = pd.to_datetime(df['TIMESTAMP'], errors='coerce', dayfirst=True)
    y = pd.to_numeric(df['Acceleration RMS'], errors='coerce')
    valid = ts.notna() & y.notna()

    if valid.sum() == 0:
        print(f'  Skipping {csv_path.name}: no valid TIMESTAMP/Acceleration RMS rows', flush=True)
        continue

    ts = ts[valid]
    y = y[valid]

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(ts, y, linewidth=1.0)
    ax.set_title(f'{csv_path.name} - Acceleration RMS vs TIMESTAMP')
    ax.set_xlabel('TIMESTAMP')
    ax.set_ylabel('Acceleration RMS')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()
